# 使用 Ohsome API 获取「真实历史」道路数据（按 grid × year）并覆盖 Excel 静态字段

本 Notebook 会把你上传的 Excel（例如 `grids_set_4_pop_masked_processed_2.xlsx`）里与年份无关、被重复填充的道路静态字段，改为**按年份**从 **Ohsome API（OSHDB/OSM 历史快照）**重新抓取并覆盖。

实现内容（对应你的需求）：

- ✅ 透过 Ohsome API 获取历史道路长度（按 *每个年份* × *每个 grid*）
- ✅ 覆盖 Excel 里原本“与年份无关”的静态道路长度字段（`len_*`）
- ✅ 可选：透过 Ohsome API 的 extraction endpoint 抓取每年道路几何，估算 `Intersec`（路口数）
- ✅ 断点续传：  
  - 道路长度（aggregation）用 **请求级缓存**（cache file）  
  - 路口数（geometry + 计算）用 **CSV 追加记录** + **GeoJSON 缓存**
- ✅ 代码可复用：统一用参数控制输入/输出文件名与列名，不再写死路径

> 注意：运行本 Notebook 需要能访问互联网（请求 `https://api.ohsome.org`）。  
> 如果在无网环境跑，所有 API 请求都会失败。

---

## 输出

- `*_ohsome_roads.xlsx`：覆盖后的主表
- （可选）`intersections_results.csv`：路口数断点续传结果
- （可选）`*_MQ.xlsx` / `*_HQ.xlsx`：按你原本规则筛选后的子集


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

# 本地可复用模块（本项目已附带）
from ohsome_road_history import (
    OhsomeClient,
    chunked,
    fetch_road_lengths_groupby_boundary_tag,
    build_road_length_table_km,
    ensure_all_grid_year,
    update_df_with_road_lengths,
    recompute_use_columns,
    compute_intersections_for_grid_years,
)

pd.set_option("display.max_columns", 200)


In [9]:
# =========================
# 1) 你只需要改这里的参数
# =========================

# 输入 Excel（可改成任何文件名）
INPUT_XLSX = Path("/Volumes/Winston_Elements/Data/Transport dynamic/grids_set_4_pop_masked.xlsx")

# 输出 Excel（默认：在原文件名后加 _ohsome_roads）
OUTPUT_XLSX = Path(f"{INPUT_XLSX.stem}_ohsome_roads.xlsx")

# 缓存目录（用于断点续传）
CACHE_DIR = Path("ohsome_cache")

# 每次 API 请求打包多少个 grid（太大可能 413；太小请求次数会变多）
CHUNK_SIZE = 300

# 把 year 映射到哪个日期快照（默认用 01-01；如果你想“年底快照”，可改 12-31）
YEAR_SNAPSHOT_MONTH_DAY = "01-01"

# Ohsome API 入口（通常不用改）
OHSOME_BASE_URL = "https://api.ohsome.org/v1"


# =========================
# 2) 你要覆盖的道路字段定义
# =========================
# Excel 里的道路长度列（单位：km）：
ROAD_LEN_COLS = ["len_mot", "len_tru", "len_pri", "len_sec", "len_ter", "len_urb", "len_unc"]

# len_* 对应哪些 highway 值（会从 ohsome 拿到每个 highway=xxx 的长度，再做求和）
ROAD_TAG_GROUPS = {
    "len_mot": ["motorway", "motorway_link"],
    "len_tru": ["trunk", "trunk_link"],
    "len_pri": ["primary", "primary_link"],
    "len_sec": ["secondary", "secondary_link"],
    "len_ter": ["tertiary", "tertiary_link"],
    # 与你原 notebook 的 urban_km 一致：residential/unclassified/service/living_street 总和
    "len_urb": ["residential", "unclassified", "service", "living_street"],
    # unclassified 单独也保留
    "len_unc": ["unclassified"],
}

# groupByValues（一次请求要取回的所有 highway 值）
GROUPBY_VALUES = sorted({v for vs in ROAD_TAG_GROUPS.values() for v in vs})

# 推荐在 filter 里也限制到同一批 highway（避免 remainder 太大）
ROAD_FILTER_EXPR = "type:way and highway in (" + ", ".join(GROUPBY_VALUES) + ")"

print("GROUPBY_VALUES =", GROUPBY_VALUES)
print("ROAD_FILTER_EXPR =", ROAD_FILTER_EXPR)


GROUPBY_VALUES = ['living_street', 'motorway', 'motorway_link', 'primary', 'primary_link', 'residential', 'secondary', 'secondary_link', 'service', 'tertiary', 'tertiary_link', 'trunk', 'trunk_link', 'unclassified']
ROAD_FILTER_EXPR = type:way and highway in (living_street, motorway, motorway_link, primary, primary_link, residential, secondary, secondary_link, service, tertiary, tertiary_link, trunk, trunk_link, unclassified)


In [10]:
# =========================
# 3) 读取 Excel，整理 grids 与 years
# =========================
df = pd.read_excel(INPUT_XLSX)

required_cols = ["grid_id", "year", "lat_min", "lon_min", "lat_max", "lon_max"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"输入表格缺少必要栏位: {missing}")

# years（排除 NaN）
years = (
    pd.to_numeric(df["year"], errors="coerce")
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
    .tolist()
)

# 每个 grid 只取一行 bbox
grids = (
    df[["grid_id", "lat_min", "lon_min", "lat_max", "lon_max"]]
    .drop_duplicates(subset=["grid_id"])
    .copy()
)
grids["grid_id"] = grids["grid_id"].astype(str)

print("rows:", df.shape[0], "unique grids:", grids.shape[0], "years:", years[:5], "...", years[-5:])
df.head()


rows: 61780 unique grids: 6178 years: [2015, 2016, 2017, 2018, 2019] ... [2020, 2021, 2022, 2023, 2024]


,grid_id,year,nation_code,lat_min,lon_min,lat_max,lon_max,cell_area,region_type,city_type,Intersec,len_mot,len_tru,len_pri,len_sec,len_ter,len_urb,distance_to_primary_rd,high_order_external_connect_index,covid_intensity,VIIRS_last_year,WorldPop_last_year,VIIRS,WorldPop,dVIIRS,dlogVIIRS,dWorldPop,City,description,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use
0,1000,2015.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.003876,929.313354,NaN,NaN,NaN,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1000,2016.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.003876,929.313354,30.521238,926.744141,-5.482637,-0.160361,-2.569214,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1000,2017.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.521238,926.744141,33.009335,922.469238,2.488096,0.075973,-4.274902,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1000,2018.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.009335,922.469238,33.678230,915.595642,0.668896,0.019477,-6.873596,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1000,2019.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.678230,915.595642,32.504826,906.136353,-1.173405,-0.034423,-9.459290,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
from tqdm.auto import tqdm


In [12]:
# =========================
# 4) 通过 Ohsome API 抓取历史道路长度（断点续传）
# =========================
# 断点续传逻辑：
# - 每个 batch 请求会在 CACHE_DIR/length_groupby/ 生成一个 json 缓存文件
# - 下次重跑时，只要缓存还在，就会直接读缓存，不会重复打 API

client = OhsomeClient(base_url=OHSOME_BASE_URL)

cache_len_dir = CACHE_DIR / "length_groupby"
cache_len_dir.mkdir(parents=True, exist_ok=True)

grid_records = grids.to_dict("records")

all_long = []

batches = list(chunked(grid_records, CHUNK_SIZE))

for i, batch in enumerate(
    tqdm(
        batches,
        total=len(batches),
        desc="Fetching road lengths (Ohsome)",
        unit="batch",
    )
):
    batch_df = pd.DataFrame(batch)

    long_df = fetch_road_lengths_groupby_boundary_tag(
        client,
        batch_df,
        years,
        month_day=YEAR_SNAPSHOT_MONTH_DAY,
        groupby_values=GROUPBY_VALUES,
        filter_expr=ROAD_FILTER_EXPR,
        cache_dir=cache_len_dir,
        timeout_param_s=400,
    )

    all_long.append(long_df)

road_long = pd.concat(all_long, ignore_index=True)
road_long.head()


Fetching road lengths (Ohsome): 100%|██████████| 21/21 [37:23<00:00, 106.82s/batch]


,grid_id,timestamp,year,tag,value_m
0,1000,2015-01-01T00:00:00Z,2015,residential,15482.60
1,1000,2016-01-01T00:00:00Z,2016,residential,15458.99
2,1000,2017-01-01T00:00:00Z,2017,residential,15458.99
3,1000,2018-01-01T00:00:00Z,2018,residential,15770.43
4,1000,2019-01-01T00:00:00Z,2019,residential,15838.97


In [13]:
# =========================
# 5) 汇总成 grid × year 的道路长度宽表（单位：km）并覆盖原 df
# =========================

road_lengths_km = build_road_length_table_km(
    road_long,
    road_tag_groups=ROAD_TAG_GROUPS,
)

# 保证所有 grid×year 都有一行（缺失的填 0）
base = ensure_all_grid_year(grids, years)
road_lengths_km = base.merge(road_lengths_km, on=["grid_id", "year"], how="left").fillna(0.0)

print("road_lengths_km shape:", road_lengths_km.shape)
road_lengths_km.head()


road_lengths_km shape: (61780, 9)


,grid_id,year,len_mot,len_tru,len_pri,len_sec,len_ter,len_urb,len_unc
0,1000,2015,2.90795,1.94627,0.0,0.0,5.42191,18.78605,0.00000
1,1000,2016,2.92063,1.94686,0.0,0.0,5.42155,24.17792,0.00000
2,1000,2017,2.92063,1.94686,0.0,0.0,5.42155,24.17792,0.00000
3,1000,2018,2.92063,1.94686,0.0,0.0,5.38919,25.06028,0.00000
4,1000,2019,2.89986,1.94682,0.0,0.0,5.38624,26.76590,0.01672


In [14]:
# 覆盖道路长度列（len_*），并重算 *_use
df2 = update_df_with_road_lengths(df, road_lengths_km, road_cols=ROAD_LEN_COLS)
df2 = recompute_use_columns(df2)

# 保存
df2.to_excel(OUTPUT_XLSX, index=False)
print("Saved:", OUTPUT_XLSX.resolve())
df2.head()


Saved: /Volumes/Winston_Elements/Data/Transport dynamic/grids_set_4_pop_masked_ohsome_roads.xlsx


,grid_id,year,nation_code,lat_min,lon_min,lat_max,lon_max,cell_area,region_type,city_type,Intersec,len_mot,len_tru,len_pri,len_sec,len_ter,len_urb,distance_to_primary_rd,high_order_external_connect_index,covid_intensity,VIIRS_last_year,WorldPop_last_year,VIIRS,WorldPop,dVIIRS,dlogVIIRS,dWorldPop,City,description,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use,len_unc
0,1000,2015,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,2.90795,1.94627,0.0,0.0,5.42191,18.78605,NaN,NaN,NaN,NaN,NaN,36.003876,929.313354,NaN,NaN,NaN,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.120158,0.749714,0.0,0.0,2.088549,7.236486,0.00000
1,1000,2016,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,2.92063,1.94686,0.0,0.0,5.42155,24.17792,NaN,NaN,NaN,36.003876,929.313354,30.521238,926.744141,-5.482637,-0.160361,-2.569214,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.125042,0.749941,0.0,0.0,2.088410,9.313463,0.00000
2,1000,2017,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,2.92063,1.94686,0.0,0.0,5.42155,24.17792,NaN,NaN,NaN,30.521238,926.744141,33.009335,922.469238,2.488096,0.075973,-4.274902,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.125042,0.749941,0.0,0.0,2.088410,9.313463,0.00000
3,1000,2018,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,2.92063,1.94686,0.0,0.0,5.38919,25.06028,NaN,NaN,NaN,33.009335,922.469238,33.678230,915.595642,0.668896,0.019477,-6.873596,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.125042,0.749941,0.0,0.0,2.075945,9.653353,0.00000
4,1000,2019,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,NaN,2.89986,1.94682,0.0,0.0,5.38624,26.76590,NaN,NaN,NaN,33.678230,915.595642,32.504826,906.136353,-1.173405,-0.034423,-9.459290,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.117041,0.749925,0.0,0.0,2.074808,10.310367,0.01672


## （可选）6) 计算历史路口数 Intersec（很慢，但可断点续传）

Ohsome API 的 aggregation endpoint **无法直接给出“路口数”**（交叉点通常不是带 tag 的 node）。  
如果你确实要像原本 OSMnx 那样的 `Intersec`，就需要：

1. 用 `/elements/geometry` 抓取每个 grid 在该年的道路几何（LineString）
2. 在本地根据几何做 `unary_union` 与端点图构建，估算 degree>=3 的节点数

这一步计算量非常大（6178 grids × 10 years ≈ 6 万次请求 + 本地几何运算）。  
所以默认关闭，你可以先跑道路长度（len_*），确认没问题后再开启。

断点续传机制：

- 每个 grid×year 会把 GeoJSON 缓存在 `CACHE_DIR/intersections_geojson/`
- 每算完一个结果就 append 到 `CACHE_DIR/intersections_results.csv`
- 中断重跑会自动跳过已完成的 (grid_id, year)


In [ ]:
COMPUTE_INTERSECTIONS = False  # <<<<<< 改成 True 才会跑

if COMPUTE_INTERSECTIONS:
    cache_geom_dir = CACHE_DIR / "intersections_geojson"
    out_csv = CACHE_DIR / "intersections_results.csv"

    inter_df = compute_intersections_for_grid_years(
        client=client,
        grids=grids,
        years=years,
        month_day=YEAR_SNAPSHOT_MONTH_DAY,
        road_filter_expr=ROAD_FILTER_EXPR,
        cache_dir=cache_geom_dir,
        out_csv=out_csv,
        snap_tolerance_m=15.0,  # 类似原 notebook 的 intersection_tolerance=15m
        min_degree=3,
        sleep_s=0.0,  # 如遇到 429 可加一点 sleep，例如 0.1
    )

    # 合并覆盖 df2 的 Intersec
    inter_df["grid_id"] = inter_df["grid_id"].astype(str)
    inter_df["year"] = pd.to_numeric(inter_df["year"], errors="coerce").astype(int)

    df3 = df2.copy()
    df3["grid_id"] = df3["grid_id"].astype(str)
    df3["year"] = pd.to_numeric(df3["year"], errors="coerce")

    df3 = df3.merge(inter_df, on=["grid_id", "year"], how="left", suffixes=("", "_new"))
    if "Intersec_new" in df3.columns:
        df3["Intersec"] = df3["Intersec_new"].fillna(df3["Intersec"]).astype(float)
        df3.drop(columns=["Intersec_new"], inplace=True)

    # 重算 Intersec_use
    df3 = recompute_use_columns(df3)

    # 另存一个包含路口数的版本
    OUTPUT_XLSX_WITH_INTERSEC = Path(f"{INPUT_XLSX.stem}_ohsome_roads_with_intersec.xlsx")
    df3.to_excel(OUTPUT_XLSX_WITH_INTERSEC, index=False)
    print("Saved with Intersec:", OUTPUT_XLSX_WITH_INTERSEC.resolve())

else:
    print("跳过 Intersec（如需运行，请把 COMPUTE_INTERSECTIONS=True）")


## 7)（可选）输出 MQ / HQ 子集（保留你原本规则）

- MQ: `Intersec + len_*` 至少 3 个非 0  
- HQ: `Intersec + len_*` 至少 4 个非 0

注意：如果你没跑 Intersec（默认关闭），MQ/HQ 就仍然使用原表的 Intersec 值（可能是静态的）。


In [16]:
QUAL_COLS = ["Intersec", "len_mot", "len_tru", "len_pri", "len_sec", "len_ter", "len_urb", "len_unc"]

# 确保是数值
tmp = df2.copy()
tmp[QUAL_COLS] = tmp[QUAL_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0)

# 按行判断 MQ/HQ 条件，再按 grid 聚合：只要同一 grid 的任一年度满足即保留整个 grid 的所有年度
mq_row = (tmp[QUAL_COLS] != 0).sum(axis=1) >= 3
hq_row = (tmp[QUAL_COLS] != 0).sum(axis=1) >= 4

# 统一用字符串形式的 grid_id 以防类型不一致
tmp["_grid_str"] = tmp["grid_id"].astype(str)

mq_grids = tmp.loc[mq_row, "_grid_str"].unique()
hq_grids = tmp.loc[hq_row, "_grid_str"].unique()

mqdf = tmp[tmp["_grid_str"].isin(mq_grids)].drop(columns=["_grid_str"])
hqdf = tmp[tmp["_grid_str"].isin(hq_grids)].drop(columns=["_grid_str"])

mq_path = Path(f"{INPUT_XLSX.stem}_MQ.xlsx")
hq_path = Path(f"{INPUT_XLSX.stem}_HQ.xlsx")

mqdf.to_excel(mq_path, index=False)
hqdf.to_excel(hq_path, index=False)

print("MQ rows:", mqdf.shape, "->", mq_path.resolve())
print("HQ rows:", hqdf.shape, "->", hq_path.resolve())


MQ rows: (39620, 37) -> /Volumes/Winston_Elements/Data/Transport dynamic/grids_set_4_pop_masked_MQ.xlsx
HQ rows: (25660, 37) -> /Volumes/Winston_Elements/Data/Transport dynamic/grids_set_4_pop_masked_HQ.xlsx


In [15]:
# Remove the 'len_unc' column if it exists
df = df.drop(columns=['len_unc'], errors='ignore')

# Assign covid_intensity values for years 2015 to 2024
covid_intensity_values = [0.00, 0.00, 0.00, 0.00, 0.05, 0.80, 1.00, 0.60, 0.20, 0.05]
years_to_update = list(range(2015, 2025))

for year, intensity in zip(years_to_update, covid_intensity_values):
    df.loc[df['year'] == year, 'covid_intensity'] = intensity

# Save the updated DataFrame back to the original file
df.to_csv('transportation_automation_set_4_ohsome.csv', index=False)

In [2]:
import pandas as pd
from pathlib import Path

# ===== 路径 =====
BASE = Path("/Volumes/Winston_Elements/Data/Transport dynamic")

STATIC_PATH = "/Volumes/Winston_Elements/Data/Transport dynamic/STA_grids_set_4_pop_masked_processed.xlsx"
MQ_PATH = "grids_set_4_pop_masked_MQ.xlsx"
HQ_PATH = "grids_set_4_pop_masked_HQ.xlsx"

# ===== 读取数据 =====
static_df = pd.read_excel(STATIC_PATH)
mq_df = pd.read_excel(MQ_PATH)
hq_df = pd.read_excel(HQ_PATH)

# ===== 构造 grid_id -> Intersec 映射 =====
intersec_map = (
    static_df
    .loc[:, ["grid_id", "Intersec"]]
    .drop_duplicates("grid_id")
    .set_index("grid_id")["Intersec"]
)

# ===== 映射到 MQ / HQ（不会改变行数） =====
mq_df["Intersec"] = mq_df["grid_id"].map(intersec_map)
hq_df["Intersec"] = hq_df["grid_id"].map(intersec_map)

# ===== 简单 sanity check =====
print("MQ Intersec missing rate:", mq_df["Intersec"].isna().mean())
print("HQ Intersec missing rate:", hq_df["Intersec"].isna().mean())

# ===== 覆盖保存（或你也可以另存） =====
mq_df.to_excel(MQ_PATH, index=False)
hq_df.to_excel(HQ_PATH, index=False)

print("Done: Intersec mapped into MQ & HQ.")


MQ Intersec missing rate: 0.0
HQ Intersec missing rate: 0.0
Done: Intersec mapped into MQ & HQ.


In [3]:
hq_df.head()

,grid_id,year,nation_code,lat_min,lon_min,lat_max,lon_max,cell_area,region_type,city_type,...,City,description,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use,len_unc
0,1000,2015.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.120158,0.749714,0.0,0.0,2.088549,7.236486,0.00000
1,1000,2016.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.125042,0.749941,0.0,0.0,2.088410,9.313463,0.00000
2,1000,2017.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.125042,0.749941,0.0,0.0,2.088410,9.313463,0.00000
3,1000,2018.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.125042,0.749941,0.0,0.0,2.075945,9.653353,0.00000
4,1000,2019.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,NaN,1.117041,0.749925,0.0,0.0,2.074808,10.310367,0.01672


In [4]:
import numpy as np

# ===== 自动识别面积列 =====
AREA_CANDIDATES = ["cell_area", "area", "area_km2"]

def find_area_col(df):
    for c in AREA_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError("No area column found (expected one of grid_area / area / area_km2)")

area_col = find_area_col(hq_df)
print("Using area column:", area_col)

# ===== 计算 Intersec_use =====
for df in [mq_df, hq_df]:
    df["Intersec_use"] = df["Intersec"] / df[area_col]
    
    # 防御性处理
    df.loc[df[area_col] <= 0, "Intersec_use"] = np.nan
    df["Intersec_use"].replace([np.inf, -np.inf], np.nan, inplace=True)

# ===== quick sanity =====
print("HQ Intersec_use describe:")
print(hq_df["Intersec_use"].describe())

print("HQ Intersec_use missing rate:",
      hq_df["Intersec_use"].isna().mean())


Using area column: cell_area
HQ Intersec_use describe:
count    25660.000000
mean        78.954033
std         58.075412
min          0.000000
25%         27.199528
50%         73.134001
75%        118.879342
max        362.569435
Name: Intersec_use, dtype: float64
HQ Intersec_use missing rate: 0.0


/var/folders/4s/v_zj4zkn43g1ssk3s0x1n33m0000gp/T/ipykernel_19651/1703455925.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Intersec_use"].replace([np.inf, -np.inf], np.nan, inplace=True)
/var/folders/4s/v_zj4zkn43g1ssk3s0x1n33m0000gp/T/ipykernel_19651/1703455925.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we ar

In [5]:
mq_df.to_excel(MQ_PATH, index=False)
hq_df.to_excel(HQ_PATH, index=False)

In [ ]:
mq_df.describe()

,grid_id,year,lat_min,lon_min,lat_max,lon_max,cell_area,Intersec,len_mot,len_tru,...,dlogVIIRS,dWorldPop,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use,len_unc
count,39620.000000,39619.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,...,35658.000000,35658.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000,39620.000000
mean,4504.108279,2019.499962,33.066532,-21.387594,33.079406,-21.374721,1.687615,105.721858,0.759234,0.357373,...,0.014246,13.591756,65.091883,0.479077,0.215767,0.597627,0.794264,0.995667,7.601784,0.800686
std,2122.784208,2.872344,15.554160,93.256918,15.554182,93.256909,0.535826,97.738686,1.956567,1.071748,...,0.118672,46.702941,57.128378,1.251034,0.636993,0.936211,1.099821,1.148391,6.189381,2.289628
min,1000.000000,2015.000000,-41.252444,-158.015752,-41.236567,-158.003303,0.538545,0.000000,0.000000,0.000000,...,-2.213083,-733.585205,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2708.000000,2017.000000,30.437966,-94.558735,30.451046,-94.547359,1.255854,23.000000,0.000000,0.000000,...,-0.042525,-0.387756,14.709038,0.000000,0.000000,0.000000,0.000000,0.011887,2.021015,0.000000
50%,4343.000000,2019.000000,36.223704,-77.432665,36.237138,-77.421010,1.664476,82.000000,0.000000,0.000000,...,0.007718,2.353348,51.274170,0.000000,0.000000,0.000000,0.397418,0.697584,6.672446,0.000000
75%,6395.000000,2022.000000,41.266225,78.038420,41.277623,78.048640,2.073462,162.000000,0.000000,0.000000,...,0.064588,14.949738,104.140866,0.000000,0.000000,0.977831,1.187980,1.448918,12.077439,0.727137
max,8491.000000,2024.000000,61.599952,175.622887,61.610589,175.636716,3.336169,636.000000,22.391460,17.798280,...,2.087595,930.869141,392.645882,14.519695,10.035150,10.094209,10.812116,12.085254,40.851725,44.473730
